In [ ]:
## EXERCISE 1 ##

In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# data input
csvname = 'breast_cancer_data.csv'
data1 = np.loadtxt(csvname,delimiter = ',')

# get input and output of dataset
x = data1[:-1,:]
y = data1[-1:,:]

In [4]:
# compute linear combination of input points
def model(x,w):
    a = w[0] + np.dot(x.T,w[1:])
    return a.T

# an implementation of the softmax cost
def softmax(w):    
    # compute the least squares cost
    cost = np.sum(np.log(1 + np.exp(-y*model(x,w))))
    return cost/float(np.size(y))

# an implementation of the perceptron cost
def perceptron(w):
    cost = np.sum(np.maximum(0,-y*model(x,w)))
    return cost/float(np.size(y))

In [5]:
def gradient_descent(g, grad_g, w, alpha, max_its):
    weight_history = [w.copy()]
    cost_history = [g(w)]
    for k in range(max_its):
        grad = grad_g(w)
        w = w - alpha*grad
        weight_history.append(w.copy())
        cost_history.append(g(w))
    return weight_history, cost_history

def grad_softmax(w):
    m = y*model(x,w)
    temp = -y/(1 + np.exp(m))
    grad_w0 = np.sum(temp)
    grad_w1 = x @ temp.T
    grad = np.vstack([grad_w0, grad_w1])
    return grad/float(np.size(y))

def grad_perceptron(w):
    m = y*model(x,w)
    mask = (m < 0).astype(float)
    temp = -y*mask
    grad_w0 = np.sum(temp)
    grad_w1 = x @ temp.T
    grad = np.vstack([grad_w0, grad_w1])
    return grad/float(np.size(y))

max_its = 1000
w = 0.1*np.random.randn(9,1)

weight_history_softmax, cost_history_softmax = gradient_descent(softmax, grad_softmax, w.copy(), 1.0, max_its)
weight_history_perceptron, cost_history_perceptron = gradient_descent(perceptron, grad_perceptron, w.copy(), 0.1, max_its)

In [6]:
### miscounts ###
def miscount(w,x,y):
    preds = np.sign(model(x,w))  # get predictions
    preds[preds == 0] = 1  # assign 0 predictions to class 1
    return int(np.sum(preds != y))  # count misclassifications

def miscount_malignant(w,x,y):
    preds = np.sign(model(x,w))
    preds[preds == 0] = 1
    mask = (y < 0)
    return int(np.sum(preds[mask] != y[mask]))

In [7]:
miscount_history_softmax = [miscount(v,x,y) for v in weight_history_softmax]
miscount_history_perceptron = [miscount(v,x,y) for v in weight_history_perceptron]

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(cost_history_softmax, label='softmax')
plt.plot(cost_history_perceptron, label='perceptron')
plt.xlabel('iteration')
plt.ylabel('cost')
plt.legend()

plt.subplot(1,2,2)
plt.plot(miscount_history_softmax, label='softmax')
plt.plot(miscount_history_perceptron, label='perceptron')
plt.xlabel('iteration')
plt.ylabel('misclassifications')
plt.legend()
plt.tight_layout()
plt.savefig('softmax_perceptron_cost_miscount.png')
plt.clf()

print('min miscount softmax:', int(np.min(miscount_history_softmax)))
print('min miscount perceptron:', int(np.min(miscount_history_perceptron)))

miscount_malignant_softmax = [miscount_malignant(v,x,y) for v in weight_history_softmax]
miscount_malignant_perceptron = [miscount_malignant(v,x,y) for v in weight_history_perceptron]
print('min malignant miscount softmax:', int(np.min(miscount_malignant_softmax)))
print('min malignant miscount perceptron:', int(np.min(miscount_malignant_perceptron)))

min miscount softmax: 21
min miscount perceptron: 19
min malignant miscount softmax: 0
min malignant miscount perceptron: 0


<Figure size 1000x400 with 0 Axes>

In [8]:
a=np.argwhere(y>0.9)
b=np.argwhere(y<-0.9)
yc=np.arange(699)
yc[a]=1
yc[b]=0

In [9]:
# define sigmoid function
def sigmoid(t):
    return 1/(1 + np.exp(-t))

# the convex cross-entropy cost function
lam = 2*10**(-3)
def cross_entropy(w):
    # compute sigmoid of model
    a = sigmoid(model(x,w))

    # compute cost of label 0 points
    ind = np.argwhere(yc == 0)
    cost = -np.sum(np.log(1 - a[:,ind]))

    # add cost on label 1 points
    ind = np.argwhere(yc==1)
    cost -= np.sum(np.log(a[:,ind]))

    # add regularizer
    cost += lam*np.sum(w[1:]**2)

    # compute cross-entropy
    return cost/float(np.size(yc))

In [12]:
yc_row = yc.reshape(1,-1)

def grad_cross_entropy(w):
    a = sigmoid(model(x,w))
    diff = a - yc_row
    grad_w0 = np.sum(diff)
    grad_w1 = x @ diff.T + 2*lam*w[1:]
    grad = np.vstack([grad_w0, grad_w1])
    return grad/float(np.size(yc_row))

w = 0.1*np.random.randn(9,1)
weight_history_log, cost_history_log = gradient_descent(cross_entropy, grad_cross_entropy, w, 0.6, max_its)

def miscount_logistic(w,x,yc_row):
    probs = sigmoid(model(x,w))  # get predicted probabilities
    preds = (probs >= 0.5).astype(int)  # assign class 1 if prob >= 0.5, else class 0
    return int(np.sum(preds != yc_row))  # count misclassifications

miscount_history_log = [miscount_logistic(v,x,yc_row) for v in weight_history_log]
print('min miscount logistic:', int(np.min(miscount_history_log)))

# make plot of weight and cost history for logistic regression
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(cost_history_log, label='logistic regression')
plt.xlabel('iteration')
plt.ylabel('cost')
plt.legend()
plt.subplot(1,2,2)
plt.plot(miscount_history_log, label='logistic regression')
plt.xlabel('iteration')
plt.ylabel('misclassifications')
plt.legend()
plt.tight_layout()
plt.savefig('logistic_cost_miscount.png')
plt.clf()

min miscount logistic: 22


<Figure size 1000x400 with 0 Axes>

In [ ]:
## EXERCISE 2 ##

In [13]:
# standard normalization function - with nan checker / filler in-er
def standard_normalizer(x):
    # compute the mean and standard deviation of the input
    x_means = np.nanmean(x,axis = 1)[:,np.newaxis]
    x_stds = np.nanstd(x,axis = 1)[:,np.newaxis]

    # check to make sure that x_stds > small threshold, for those not
    # divide by 1 instead of original standard deviation
    ind = np.argwhere(x_stds < 10**(-2))
    if len(ind) > 0:
        ind = [v[0] for v in ind] # Just keep the row index
        adjust = np.zeros((x_stds.shape))
        adjust[ind] = 1.0
        x_stds += adjust

    # fill in any nan values with means
    ind = np.argwhere(np.isnan(x) == True)
    for i in ind:
        x[i[0],i[1]] = x_means[i[0]]

    # create standard normalizer function
    normalizer = lambda data: (data - x_means)/x_stds

    # create inverse standard normalizer
    inverse_normalizer = lambda data: data*x_stds + x_means

    # return normalizer
    return normalizer,inverse_normalizer

In [14]:
# data input
csvname = 'spambase_data.csv'
data = np.loadtxt(csvname,delimiter = ',')

# get input and output of dataset
x = data[:-1,:]
y = data[-1:,:] 
normalizer,inverse_normalizer = standard_normalizer(x)
x = normalizer(x)

In [16]:
# Exercise 2: softmax vs perceptron on spambase
N = x.shape[0]
w = 0.1*np.random.randn(N+1,1)

weight_history_softmax2, cost_history_softmax2 = gradient_descent(softmax, grad_softmax, w.copy(), 1.0, max_its)
weight_history_perceptron2, cost_history_perceptron2 = gradient_descent(perceptron, grad_perceptron, w.copy(), 0.1, max_its)

miscount_history_softmax2 = [miscount(v,x,y) for v in weight_history_softmax2]
miscount_history_perceptron2 = [miscount(v,x,y) for v in weight_history_perceptron2]

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(cost_history_softmax2, label='softmax')
plt.plot(cost_history_perceptron2, label='perceptron')
plt.xlabel('iteration')
plt.ylabel('cost')
plt.legend()

plt.subplot(1,2,2)
plt.plot(miscount_history_softmax2, label='softmax')
plt.plot(miscount_history_perceptron2, label='perceptron')
plt.xlabel('iteration')
plt.ylabel('misclassifications')
plt.legend()
plt.tight_layout()
plt.savefig('softmax_perceptron_spambase_cost_miscount.png')
plt.clf()

min_mis_soft = int(np.min(miscount_history_softmax2))  # minimum misclassification count for softmax
min_mis_perc = int(np.min(miscount_history_perceptron2))  # minimum misclassification count for perceptron
P = y.size
acc_soft = 1 - min_mis_soft/float(P)  # accuracy for softmax
acc_perc = 1 - min_mis_perc/float(P)  # accuracy for perceptron
print('min miscount softmax:', min_mis_soft)
print('min miscount perceptron:', min_mis_perc)
print('accuracy softmax:', acc_soft)
print('accuracy perceptron:', acc_perc)

# confusion matrix for best softmax weights
best_idx = int(np.argmin(miscount_history_softmax2))  # index of the best softmax weights
w_best = weight_history_softmax2[best_idx]  # best softmax weights
preds = np.sign(model(x,w_best))  # get predictions for best softmax weights
preds[preds == 0] = 1  # assign 0 predictions to class 1
tp = int(np.sum((y==1) & (preds==1)))  # true positives: predicted 1 and true label is 1
fn = int(np.sum((y==1) & (preds==-1)))  # false negatives: predicted -1 but true label is 1
fp = int(np.sum((y==-1) & (preds==1)))  # false positives: predicted 1 but true label is -1
tn = int(np.sum((y==-1) & (preds==-1)))  # true negatives: predicted -1 and true label is -1
conf_mat = np.array([[tp, fn],[fp, tn]])
print('confusion matrix (rows: true [1,-1], cols: pred [1,-1]):')
print(conf_mat)

min miscount softmax: 331
min miscount perceptron: 318
accuracy softmax: 0.9280591175831341
accuracy perceptron: 0.9308845903064551
confusion matrix (rows: true [1,-1], cols: pred [1,-1]):
[[1608  205]
 [ 126 2662]]


<Figure size 1000x400 with 0 Axes>

In [ ]:
## EXERCISE 3 ##

In [23]:
# Exercise 3: fetch datasets from UCI
from ucimlrepo import fetch_ucirepo

breastcancer = fetch_ucirepo(id=15)
X = breastcancer.data.features
y = breastcancer.data.targets

In [24]:
# 3.1 drop column 'Single_epithelial_cell_size'
X = X.drop(columns=['Single_epithelial_cell_size'])

In [25]:
# 3.2 replace labels: 2 -> 1 (benign), 4 -> -1 (malignant)
y = y.replace({2: 1, 4: -1})

In [ ]:
## EXERCISE 4 ##

In [17]:
# load in dataset
csvname = 'credit_dataset.csv'
data = np.loadtxt(csvname,delimiter = ',')
x = data[:-1,:]
y = data[-1:,:]

In [18]:
# Exercise 4.1: standard normalize inputs
normalizer, inverse_normalizer = standard_normalizer(x)
x = normalizer(x)

In [19]:
# Exercise 4.2: perceptron fit with gradient descent
N = x.shape[0]
w = 0.1*np.random.randn(N+1,1)
weight_history_credit, cost_history_credit = gradient_descent(perceptron, grad_perceptron, w, 0.1, max_its)

miscount_history_credit = [miscount(v,x,y) for v in weight_history_credit]

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(cost_history_credit)
plt.xlabel('iteration')
plt.ylabel('cost')

plt.subplot(1,2,2)
plt.plot(miscount_history_credit)
plt.xlabel('iteration')
plt.ylabel('misclassifications')
plt.tight_layout()
plt.savefig('credit_perceptron_cost_miscount.png')
plt.clf()

min_mis = int(np.min(miscount_history_credit))
acc = 1 - min_mis/float(y.size)
print('min miscount:', min_mis)
print('accuracy:', acc)

min miscount: 233
accuracy: 0.767


<Figure size 1000x400 with 0 Axes>

In [30]:
# Exercise 4.3: confusion matrix for best perceptron weights
best_idx = int(np.argmin(miscount_history_credit))
w_best = weight_history_credit[best_idx]
preds = np.sign(model(x,w_best))
preds[preds == 0] = 1
tp = int(np.sum((y==1) & (preds==1)))
fn = int(np.sum((y==1) & (preds==-1)))
fp = int(np.sum((y==-1) & (preds==1)))
tn = int(np.sum((y==-1) & (preds==-1)))
conf_mat = np.array([[tp, fn],[fp, tn]])
print('confusion matrix (rows: true [1,-1], cols: pred [1,-1]):')
print(conf_mat)

confusion matrix (rows: true [1,-1], cols: pred [1,-1]):
[[607  93]
 [148 152]]
